# Reduced Inequality: Socio-Economic Data Mining Analysis (UN SDG 10)
**Nişantaşı University — Data Mining Course Project Exhibition 2024**

**Author:** Irmak Kartalkayası  
**Data Sources:** TUIK (Turkish Statistical Institute) & OECD Disparity Database (2015–2024)

---
### Project Summary
This notebook implements an end-to-end data mining and machine learning pipeline to investigate multi-dimensional inequality across Turkey and selected European OECD countries (Germany, Italy, France, Spain, Netherlands).

Key analytical components include:
1. **Exploratory Data Analysis (EDA)** & Descriptive Statistics
2. **Supervised Learning**: Ordinary Least Squares (OLS) Linear Regression modeling correlations between poverty rates and gender wage disparity
3. **Unsupervised Learning**: K-Means Clustering on normalized indicators with Elbow & Silhouette analysis
4. **Policy Recommendations & Academic Synthesis**

## 1. Setup & Environment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Data Loading & Inspection

In [ ]:
# Load the cleaned dataset
df = pd.read_csv("../data/inequality_data.csv")
print(f"Dataset shape: {df.shape}")
df.head(10)

In [ ]:
# Statistical Summary
df.describe().T

## 3. Exploratory Data Analysis & Visualization

In [ ]:
# Correlation Heatmap
features = [
    "gini_coefficient", "poverty_rate_pct", "income_disparity_ratio",
    "gender_wage_gap_pct", "education_quality_index", "unemployment_disparity"
]
plt.figure(figsize=(9, 7))
sns.heatmap(df[features].corr(), annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Matrix of Socio-Economic Indicators", fontsize=14, pad=12)
plt.show()

In [ ]:
# Historical Gini Index Trends across Countries
plt.figure(figsize=(11, 6))
for country in df["country"].unique():
    subset = df[df["country"] == country].sort_values("year")
    plt.plot(subset["year"], subset["gini_coefficient"], marker="o", linewidth=2, label=country)

plt.title("Evolution of Gini Coefficient (2015 - 2024)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Gini Coefficient")
plt.legend(title="Country")
plt.show()

## 4. Supervised Learning: Linear Regression Analysis
Examining how changes in the **Poverty Rate** correlate with and predict the **Gender Wage Gap**.

In [ ]:
X = df[["poverty_rate_pct"]].values
y = df["gender_wage_gap_pct"].values

lr = LinearRegression()
lr.fit(X, y)
y_pred = lr.predict(X)

r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"Equation: Gender Wage Gap = {lr.coef_[0]:.4f} * Poverty Rate + {lr.intercept_:.4f}")
print(f"R² Score: {r2:.4f}")
print(f"RMSE:     {rmse:.4f}")

In [ ]:
# Plot regression line
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="poverty_rate_pct", y="gender_wage_gap_pct", hue="country", s=100)
x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(x_line, lr.predict(x_line), color="crimson", linewidth=2.5, label=f"Fit: R²={r2:.2f}")
plt.title("Linear Regression: Poverty Rate vs. Gender Wage Gap", fontsize=14)
plt.xlabel("Poverty Rate (%)")
plt.ylabel("Gender Wage Gap (%)")
plt.legend()
plt.show()

## 5. Unsupervised Learning: K-Means Clustering
Segmenting socio-economic profiles using multi-attribute clustering to detect systemic inequality groups.

In [ ]:
cluster_features = [
    "gini_coefficient",
    "poverty_rate_pct",
    "income_disparity_ratio",
    "gender_wage_gap_pct",
    "education_quality_index"
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[cluster_features])

# Find optimal K via Elbow Method
inertias = []
ks = range(2, 7)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(ks, inertias, marker="o", color="darkblue")
plt.title("Elbow Method for Optimal K", fontsize=13)
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.show()

In [ ]:
# Train final KMeans with K=3
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, df["cluster"])
print(f"Silhouette Score: {sil:.3f}")

# Cluster profiles summary
df.groupby("cluster")[cluster_features].mean()

In [ ]:
# Visualize Clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="gini_coefficient",
    y="poverty_rate_pct",
    hue="cluster",
    style="country",
    palette="Set1",
    s=120
)
plt.title("K-Means Clusters: Gini Index vs. Poverty Rate", fontsize=14)
plt.xlabel("Gini Coefficient")
plt.ylabel("Poverty Rate (%)")
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.tight_layout()
plt.show()

## 6. Key Conclusions & Findings
1. **High Disparity Clusters**: Countries with higher Gini coefficients consistently exhibit wider urban-rural unemployment gaps and higher income disparity ratios.
2. **Gender Wage Gap & Poverty Dynamics**: Inverse correlation trends highlight structural nuances where lower informal female labor participation in certain emerging economies alters the observable wage gap metrics.
3. **Policy Implications**: Targeted educational investments and rural infrastructural support demonstrate the highest leverage for reducing systemic inequalities in accordance with UN SDG 10.